##S11 - Setting up the pipeline to collect information on contract data from three different source:

a. Rookie Scale from RealGM at season x draft pick grain
b. Minimum salary scale from RealGM at season x years of service grain
c. Contract details for non rookie deals and non minimum contract i.e all other market contracts for NBA players

### Rookie Scale Code

In [48]:
import pandas as pd
import os
import time

rookie_raw = pd.read_csv("../data/raw/contracts/Rookie Deal Salary Scale.csv")

rookie_raw.shape

(800, 8)

In [49]:
rookie_raw.head()

,Pick,1st Year Salary,2nd Year Salary,3rd Year Option Salary,4th Year Option:\nPercentage Increased Over 3rd Year Salary,Qualifying Offer:\nPercentage Increase Over 4th Year Salary,Season,Year
0,1,"$11,521,600","$12,097,900","$12,673,900",26.10%,40.00%,2025-2026,2026.0
1,2,"$10,308,600","$10,824,200","$11,339,800",26.20%,40.50%,2025-2026,2026.0
2,3,"$9,257,400","$9,719,900","$10,183,200",26.40%,41.20%,2025-2026,2026.0
3,4,"$8,346,400","$8,763,800","$9,181,300",26.50%,41.90%,2025-2026,2026.0
4,5,"$7,558,200","$7,935,900","$8,313,800",26.70%,42.60%,2025-2026,2026.0


In [50]:
rookie_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 800 entries, 0 to 799
Data columns (total 8 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Pick                                                        800 non-null    str    
 1   1st Year Salary                                             800 non-null    str    
 2   2nd Year Salary                                             800 non-null    str    
 3   3rd Year Option Salary                                      800 non-null    str    
 4   4th Year Option:
Percentage Increased Over 3rd Year Salary  800 non-null    str    
 5   Qualifying Offer:
Percentage Increase Over 4th Year Salary  800 non-null    str    
 6   Season                                                      775 non-null    str    
 7   Year                                                        775 non-null    float64
dtypes: float64(1

In [51]:
rookie_raw["Pick"].value_counts()

Pick
1       26
2       26
3       26
4       26
5       26
6       26
7       26
8       26
9       26
10      26
11      26
12      26
13      26
14      26
15      26
16      26
17      26
18      26
19      26
20      26
21      26
22      26
23      26
24      26
25      26
26      26
27      26
28      26
29      26
Pick    25
30      21
Name: count, dtype: int64

In [52]:
rookie_raw.duplicated(subset=["Season", "Pick"]).sum()

np.int64(24)

In [53]:
rookie_scale_raw = rookie_raw[rookie_raw["Year"].notna()].copy()

rookie_scale_raw.shape

(775, 8)

In [54]:
rookie_scale_raw.info()

<class 'pandas.DataFrame'>
Index: 775 entries, 0 to 799
Data columns (total 8 columns):
 #   Column                                                      Non-Null Count  Dtype  
---  ------                                                      --------------  -----  
 0   Pick                                                        775 non-null    str    
 1   1st Year Salary                                             775 non-null    str    
 2   2nd Year Salary                                             775 non-null    str    
 3   3rd Year Option Salary                                      775 non-null    str    
 4   4th Year Option:
Percentage Increased Over 3rd Year Salary  775 non-null    str    
 5   Qualifying Offer:
Percentage Increase Over 4th Year Salary  775 non-null    str    
 6   Season                                                      775 non-null    str    
 7   Year                                                        775 non-null    float64
dtypes: float64(1), st

In [55]:
new_columns = [
    "draft_pick",
    "year_1_salary",
    "year_2_salary",
    "year_3_option_salary",
    "year_4_option_increase_pct",
    "qualifying_offer_increase_pct",
    "season",
    "year"
]

rookie_scale_raw.columns = new_columns

rookie_scale_raw.head()

,draft_pick,year_1_salary,year_2_salary,year_3_option_salary,year_4_option_increase_pct,qualifying_offer_increase_pct,season,year
0,1,"$11,521,600","$12,097,900","$12,673,900",26.10%,40.00%,2025-2026,2026.0
1,2,"$10,308,600","$10,824,200","$11,339,800",26.20%,40.50%,2025-2026,2026.0
2,3,"$9,257,400","$9,719,900","$10,183,200",26.40%,41.20%,2025-2026,2026.0
3,4,"$8,346,400","$8,763,800","$9,181,300",26.50%,41.90%,2025-2026,2026.0
4,5,"$7,558,200","$7,935,900","$8,313,800",26.70%,42.60%,2025-2026,2026.0


In [56]:
rookie_scale_raw["year_1_salary"] = (
    rookie_scale_raw["year_1_salary"]
    .str.replace(r"[$,]", "", regex=True)
    .astype(int)
)

rookie_scale_raw["year_2_salary"] = (
    rookie_scale_raw["year_2_salary"]
    .str.replace(r"[$,]", "", regex=True)
    .astype(int)
)

rookie_scale_raw["year_3_option_salary"] = (
    rookie_scale_raw["year_3_option_salary"]
    .str.replace(r"[$,]", "", regex=True)
    .astype(int)
)

rookie_scale_raw["year_4_option_increase_pct"] = (
    rookie_scale_raw["year_4_option_increase_pct"]
    .str.replace("%", "", regex=True)
    .astype(float) 
) / 100

rookie_scale_raw["qualifying_offer_increase_pct"] = (
    rookie_scale_raw["qualifying_offer_increase_pct"]
    .str.replace("%", "", regex=True)
    .astype(float) 
) / 100

rookie_scale_raw.head()


,draft_pick,year_1_salary,year_2_salary,year_3_option_salary,year_4_option_increase_pct,qualifying_offer_increase_pct,season,year
0,1,11521600,12097900,12673900,0.261,0.400,2025-2026,2026.0
1,2,10308600,10824200,11339800,0.262,0.405,2025-2026,2026.0
2,3,9257400,9719900,10183200,0.264,0.412,2025-2026,2026.0
3,4,8346400,8763800,9181300,0.265,0.419,2025-2026,2026.0
4,5,7558200,7935900,8313800,0.267,0.426,2025-2026,2026.0


In [57]:
rookie_scale_raw["draft_pick"] = rookie_scale_raw["draft_pick"].astype(int)

rookie_scale_raw["year"] = rookie_scale_raw["year"].astype(int)

rookie_scale_raw.head()

,draft_pick,year_1_salary,year_2_salary,year_3_option_salary,year_4_option_increase_pct,qualifying_offer_increase_pct,season,year
0,1,11521600,12097900,12673900,0.261,0.400,2025-2026,2026
1,2,10308600,10824200,11339800,0.262,0.405,2025-2026,2026
2,3,9257400,9719900,10183200,0.264,0.412,2025-2026,2026
3,4,8346400,8763800,9181300,0.265,0.419,2025-2026,2026
4,5,7558200,7935900,8313800,0.267,0.426,2025-2026,2026


In [58]:
rookie_scale_raw.info()

<class 'pandas.DataFrame'>
Index: 775 entries, 0 to 799
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   draft_pick                     775 non-null    int64  
 1   year_1_salary                  775 non-null    int64  
 2   year_2_salary                  775 non-null    int64  
 3   year_3_option_salary           775 non-null    int64  
 4   year_4_option_increase_pct     775 non-null    float64
 5   qualifying_offer_increase_pct  775 non-null    float64
 6   season                         775 non-null    str    
 7   year                           775 non-null    int64  
dtypes: float64(2), int64(5), str(1)
memory usage: 54.5 KB


In [59]:
rookie_scale = rookie_scale_raw.copy()

rookie_scale.info()

<class 'pandas.DataFrame'>
Index: 775 entries, 0 to 799
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   draft_pick                     775 non-null    int64  
 1   year_1_salary                  775 non-null    int64  
 2   year_2_salary                  775 non-null    int64  
 3   year_3_option_salary           775 non-null    int64  
 4   year_4_option_increase_pct     775 non-null    float64
 5   qualifying_offer_increase_pct  775 non-null    float64
 6   season                         775 non-null    str    
 7   year                           775 non-null    int64  
dtypes: float64(2), int64(5), str(1)
memory usage: 54.5 KB


In [60]:
# 1. Historical coverage
print(rookie_scale["year"].nunique())
print(rookie_scale["year"].min(), rookie_scale_raw["year"].max())

# 2. Primary-key uniqueness
print(
    rookie_scale.duplicated(
        subset=["year", "draft_pick"]
    ).sum()
)

# 3. Draft pick counts per season
display(
    rookie_scale.groupby("year")["draft_pick"]
    .agg(["count", "min", "max"])
)

# 4. Numeric sanity checks
display(
    rookie_scale[
        [
            "year_1_salary",
            "year_2_salary",
            "year_3_option_salary",
            "year_4_option_increase_pct",
            "qualifying_offer_increase_pct"
        ]
    ].describe()
)

26
2001 2026
0


,count,min,max
year,,,
2001,29,1,29
2002,29,1,29
2003,29,1,29
2004,29,1,29
2005,29,1,29
2006,30,1,30
2007,30,1,30
2008,30,1,30
2009,30,1,30


,year_1_salary,year_2_salary,year_3_option_salary,year_4_option_increase_pct,qualifying_offer_increase_pct
count,7.750000e+02,7.750000e+02,7.750000e+02,775.000000,775.000000
mean,2.273809e+06,2.426283e+06,2.567610e+06,0.498804,0.412243
std,1.682938e+06,1.781945e+06,1.874085e+06,0.206091,0.069188
min,5.892000e+05,6.333000e+05,6.775000e+05,0.261000,0.300000
25%,1.099100e+06,1.170650e+06,1.236600e+06,0.272000,0.355000
50%,1.756000e+06,1.887700e+06,2.010700e+06,0.533000,0.412000
75%,2.817000e+06,3.026000e+06,3.218200e+06,0.697000,0.462000
max,1.152160e+07,1.209790e+07,1.267390e+07,0.805000,0.600000


In [61]:
import os

staging_dir = "../data/staging/contracts"
os.makedirs(staging_dir, exist_ok=True)

staging_path = os.path.join(
    staging_dir,
    "s11a_rookie_salary_scale.csv"
)

rookie_scale.to_csv(staging_path, index=False)

In [62]:
rookie_scale_check = pd.read_csv("../data/staging/contracts/s11a_rookie_salary_scale.csv")

rookie_scale_check.shape

(775, 8)